# QDock on a Coherent Ising Machine — three GPM redockings

Reproduce the molecular-docking results in **Wei *et al.*, "A versatile coherent
Ising computing platform," *Light: Science & Applications* (2026) 15:74**
(`s41377-025-02178-1`): three protein–ligand complexes —
PDB **1N2J**, **1LRH**, **1JD0** — docked by **Grid Point Matching (GPM)**.
Docking is cast as a **QUBO** and solved on a **real Coherent Ising Machine
(CIM)** through the **Kaiwu SDK** (开物, QBoson).

**Every Kaiwu call is shown and explained** below:

| step | Kaiwu API |
|---|---|
| init license | `kw.license.init` |
| QUBO → Ising | `kw.conversion.qubo_matrix_to_ising_matrix` |
| submit to the CIM (quota mode) | `kw.cim.CIMOptimizer(..., task_mode="quota")` |
| quantize to 8 bits | `kw.cim.PrecisionReducer(..., precision=8, truncated_precision=8)` |
| solve | `reducer.solve(ising)` |
| classical cross-check | `kw.classical.SimulatedAnnealingOptimizer` |

**The recipe used throughout: quota mode, `precision=8`, `truncated_precision=8`** —
an 8-bit truncation with **no spin expansion**, so the number of spins equals the
number of QUBO variables (232 / 304 / 244 here), all under the ~1000-spin budget.

The notebook ships the **real CIM solutions** (in `results/`) so it renders end to
end with no license; set `QDOCK_FORCE_LIVE=1` with your own credentials to resubmit
to the hardware.

## 0. Setup

Python **3.10** (required by the Kaiwu wheel). From the repo root:

```bash
python3.10 -m venv .venv && source .venv/bin/activate
python -m pip install -r requirements.txt
python -m pip install vendor/kaiwu-1.3.1-cp310-none-any.whl
```

Rebuilding a QUBO from scratch (optional) also needs the `autogrid4` and `obabel`
command-line tools (conda-forge); the prebuilt QUBOs in `results/` make that
unnecessary to *run* this notebook.

In [1]:
import os, sys
import numpy as np
from types import SimpleNamespace

# Find the repo root whether this runs from notebooks/ (Jupyter) or the repo
# root (Colab after `git clone` + `%cd`), then make `qdock_kaiwu` importable.
ROOT = None
for cand in ("..", ".", os.path.expanduser("~/qdock-kaiwu-workshop")):
    if os.path.isdir(os.path.join(cand, "results")) and os.path.isdir(os.path.join(cand, "qdock_kaiwu")):
        ROOT = os.path.abspath(cand); break
assert ROOT, "run this from the repo (could not locate results/ and qdock_kaiwu/)"
sys.path.insert(0, ROOT)
RESULTS = os.path.join(ROOT, "results")

import kaiwu as kw
from qdock_kaiwu import backends, evaluate
from qdock_kaiwu.gpm import _matches_to_poses
print("repo root :", ROOT)
print("kaiwu     :", kw.__version__)

repo root : /Users/songxinwei/Downloads/qdock-kaiwu-workshop
kaiwu     : 1.3.1


## 1. Kaiwu license — credentials from the environment, CIM off the proxy

`kw.license.init` authenticates the SDK. Credentials come from the environment
(`KAIWU_USER_ID` / `KAIWU_SDK_CODE`, free from [platform.qboson.com](https://platform.qboson.com)) —
**never hard-coded into the repo**:

```bash
export KAIWU_USER_ID=<numeric id>
export KAIWU_SDK_CODE=<sdk code>
```

The CIM is a cloud service reached over TLS; a **local HTTP/SOCKS proxy (e.g. a VPN
in TUN mode) breaks the connection**, so we clear the proxy variables first. Without
a license the notebook still runs from the cached CIM solutions in `results/`.

In [2]:
import tempfile

# CIM traffic must NOT go through a proxy.
os.environ["no_proxy"] = os.environ["NO_PROXY"] = "*"
for k in ("http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY", "all_proxy", "ALL_PROXY"):
    os.environ.pop(k, None)

# Any CIMOptimizer needs a checkpoint directory (it caches finished tasks there).
kw.common.CheckpointManager.save_dir = os.path.join(tempfile.gettempdir(), "qdock_cim_cache")
FORCE_LIVE = bool(int(os.environ.get("QDOCK_FORCE_LIVE", "0")))   # 1 = resubmit to the hardware

HAVE_LICENSE = False
try:
    kw.license.init(user_id=os.environ["KAIWU_USER_ID"], sdk_code=os.environ["KAIWU_SDK_CODE"])
    HAVE_LICENSE = True
    print("license OK — live CIM / SA available; FORCE_LIVE =", FORCE_LIVE)
except Exception as e:
    print("no license in env:", str(e)[:70])
    print("-> rendering from the cached real-CIM solutions in results/")

license OK — live CIM / SA available; FORCE_LIVE = False


## 2. The Grid Point Matching (GPM) QUBO

One binary variable per candidate match, `x[a,s] = 1` ⇔ *ligand atom `a` sits at
grid point `s`*. The energy minimised is

$$H(x)=\sum_{v} w_v\,x_v \;+\; K_{\text{dist}}\!\!\sum_{p<q}\![\,\lVert d^{\text{lig}}_{pq}-d^{\text{site}}_{pq}\rVert>c\,]\,x_px_q \;+\; K_{\text{mono}}\!\!\sum_{p<q}\![\,\text{same atom}\,]\,x_px_q$$

- **reward `w`** — the AutoGrid van-der-Waals energy of placing that atom type at
  that grid point (favourable placements are negative);
- **`K_dist`** — penalises any pair of matches that distorts the rigid ligand
  (inter-atom distance must survive, within tolerance `c`);
- **`K_mono`** — forbids one atom from occupying two grid points.

Minimising `H` selects a consistent set of matches; superposing the matched atoms
onto their grid points (Kabsch) yields a 3-D pose. The penalty weights used for
these three targets are `c = 2.565`, `K_dist = 2.337`, `K_mono = 39.530`.

**How each `results/<pdb>_cim.npz` was built** (needs `autogrid4` + the receptor;
shipped prebuilt so this notebook needs neither):

```python
from qdock_kaiwu import GPMDock, io
from qdock_kaiwu.qubo import build_gpm_qubo
g = GPMDock(backend="cim", workdir="run")
g.receptor_name = "receptor"
g.receptor_pdbqt = "data/1N2J_receptor.pdbqt"          # meeko-prepared receptor
g.receptor_ad_types = np.unique(io.read_pdbqt(g.receptor_pdbqt)["ad_types"])
g.make_ligand(["data/1N2J_ligand.mol2"])               # crystal ligand = redock reference
g.make_box_ligand("data/1N2J_ligand.mol2", center_length=6.0, grid_length=2.0, cutoff=0.0)
lig = g.ligands[0]
Q, variables = build_gpm_qubo(lig.coords, lig.ad_types, g.grid_dict, g.box_coords,
                              2.565, 2.337, 39.530)     # the GPM QUBO matrix
```

In [3]:
MOLS  = ["1N2J", "1LRH", "1JD0"]
PAPER = {"1N2J": 0.8, "1LRH": 1.4, "1JD0": 0.6}        # paper mRMSD (Angstrom)

B = {}
for p in MOLS:
    B[p] = dict(np.load(os.path.join(RESULTS, f"{p}_cim.npz"), allow_pickle=True))
    d = B[p]
    print(f"{p}: QUBO {d['Q'].shape}  ->  {int(d['n_vars'])} variables (= spins), "
          f"grid {float(d['grid_length'])} A, {len(d['solutions'])} cached CIM solutions")

1N2J: QUBO (232, 232)  ->  232 variables (= spins), grid 2.0 A, 59 cached CIM solutions
1LRH: QUBO (304, 304)  ->  304 variables (= spins), grid 2.0 A, 54 cached CIM solutions
1JD0: QUBO (244, 244)  ->  244 variables (= spins), grid 2.5 A, 17 cached CIM solutions


## 3. Walkthrough on 1N2J — every Kaiwu call, one at a time

### 3.1 QUBO → Ising

Kaiwu minimises an **Ising** Hamiltonian over ±1 spins. `qubo_matrix_to_ising_matrix`
appends one **ancilla spin** and bakes in the sign, so feeding its output to Kaiwu's
(maximising) solver *minimises* the QUBO — we never add a manual minus. A spin
configuration maps back to binary as `x_i = (1 + s_i · s_ancilla) / 2`.

In [4]:
p = "1N2J"
d = B[p]
Q = d["Q"].astype(float)
ising, offset = kw.conversion.qubo_matrix_to_ising_matrix(Q)
print(f"QUBO {Q.shape}  ->  Ising {ising.shape}  (+1 ancilla spin),  offset {offset:.3f}")

QUBO (232, 232)  ->  Ising (233, 233)  (+1 ancilla spin),  offset 31956.947


### 3.2 Submit to the CIM — **quota** mode + an **8-bit** `PrecisionReducer`

Two Kaiwu objects do the solve:

- **`kw.cim.CIMOptimizer(..., task_mode="quota")`** submits the problem to the
  photonic machine. `quota` returns a handful of the best *distinct* solutions per
  run — best single-pose accuracy on these small, dense matrices. (`task_mode="sample"`
  returns many reads; it only pays off on larger matrices.)
- **`kw.cim.PrecisionReducer(opt, precision=8, truncated_precision=8)`** maps the
  real-valued Ising matrix onto the **8-bit** hardware. `truncated_precision=8`
  truncates straight to 8 bits with **no spin expansion**, so `#spins == #variables`
  (232 for 1N2J). A larger `truncated_precision` would split one variable across
  several spins to preserve precision, which on this wide-dynamic-range docking
  matrix (`K_mono` ≫ vdW) explodes the spin count past the machine budget.

The helper below is the entire CIM solve — every Kaiwu call, commented. It returns
the shipped real-CIM solutions unless `QDOCK_FORCE_LIVE=1` (and a license) resubmits
to the hardware.

In [5]:
# The entire CIM solve: quota mode + 8-bit PrecisionReducer (no spin expansion).
def solve_quota_8bit(Q, task_name, cache=None, sample_number=300):
    if cache is not None and not (HAVE_LICENSE and FORCE_LIVE):
        return list(cache)                                              # shipped real-CIM output
    ising, _ = kw.conversion.qubo_matrix_to_ising_matrix(Q)            # 1. QUBO -> Ising
    opt = kw.cim.CIMOptimizer(task_name=task_name, wait=True, interval=2,
                              task_mode="quota", sample_number=sample_number)  # 2. CIM, quota
    reducer = kw.cim.PrecisionReducer(opt, precision=8, truncated_precision=8,
                                      only_feasible_solution=False)     # 3. 8-bit, no expansion
    spins = np.asarray(reducer.solve(ising))                           # 4. submit -> poll -> spins
    return [backends._spins_to_binary(s, Q.shape[0]) for s in spins]   #    ±1 spins -> binary

raw = solve_quota_8bit(Q, "qdock_1N2J_GPM_quota_p8t8", cache=d["solutions"])
print(f"{len(raw)} real-CIM solutions for {p}"
      f"{' (live)' if HAVE_LICENSE and FORCE_LIVE else ' (cached)'}")

59 real-CIM solutions for 1N2J (cached)


### 3.3 Decode the spins into a pose

Gauge the spins back to binary, keep the distinct assignments sorted by QUBO energy
(`_rank_unique`), then turn each into a 3-D pose by Kabsch-superposing the matched
ligand atoms onto their grid points (`_matches_to_poses`). The CIM is **high-variance**
on these dense QUBOs — the native pose surfaces in roughly one run in five — so the
shipped solutions **pool several runs** and we take the minimum-RMSD pose.

In [6]:
ranked = backends._rank_unique(raw, Q)                             # dedupe + sort by QUBO energy
lig = SimpleNamespace(coords=d["lig_coords"])                      # decode needs only .coords
poses, energies = _matches_to_poses(lig, d["variables"], d["box_coords"], ranked)
rmsds = evaluate.pose_rmsds(np.array(poses), d["lig_coords"], d["lig_elements"])
print(f"{p}: {len(ranked)} unique solutions -> {len(poses)} poses; "
      f"mRMSD = {rmsds.min():.2f} A   (paper {PAPER[p]} A)")

1N2J: 59 unique solutions -> 50 poses; mRMSD = 0.41 A   (paper 0.8 A)


## 4. All three molecules — the same quota + 8-bit recipe

`mRMSD` = the minimum heavy-atom RMSD over the sampled pose pool (the paper's
"sampling power"). The three targets are unrelated proteins with completely
different binding pockets, so each is an independent redocking test.

In [7]:
def dock(p):
    d = B[p]; Q = d["Q"].astype(float)
    raw = solve_quota_8bit(Q, f"qdock_{p}_GPM_quota_p8t8", cache=d["solutions"])
    ranked = backends._rank_unique(raw, Q)
    lig = SimpleNamespace(coords=d["lig_coords"])
    poses, _ = _matches_to_poses(lig, d["variables"], d["box_coords"], ranked)
    r = evaluate.pose_rmsds(np.array(poses), d["lig_coords"], d["lig_elements"])
    return d, len(poses), r

print(f"{'PDB':<6}{'spins':>7}{'grid/A':>8}{'poses':>7}{'mRMSD/A':>9}{'paper/A':>9}  hit")
print("-" * 54)
BEST = {}
for p in MOLS:
    d, npose, r = dock(p)
    BEST[p] = (d, r)
    flag = "<=paper" if r.min() <= PAPER[p] + 1e-9 else ("<2A" if r.min() < 2 else "")
    print(f"{p:<6}{int(d['n_vars']):>7}{float(d['grid_length']):>8.1f}{npose:>7}"
          f"{r.min():>9.2f}{PAPER[p]:>9.1f}  {flag}")

PDB     spins  grid/A  poses  mRMSD/A  paper/A  hit
------------------------------------------------------
1N2J      232     2.0     50     0.41      0.8  <=paper
1LRH      304     2.0     38     0.74      1.4  <=paper
1JD0      244     2.5     14     1.99      0.6  <2A


## 5. Best pose vs crystal

Heavy-atom overlay of the lowest-RMSD CIM pose (coloured sticks) on the crystal
ligand (blue). Saved to `assets/docking_demo.png`.

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

EL = {'C': '#444', 'N': '#2c5fa8', 'O': '#cc3333', 'S': '#d4a000', 'P': '#d4a000',
      'ZN': '#9a6', 'CL': '#3a3'}

def bonds(xyz, el):
    keep = np.array([e != 'H' for e in el]); idx = np.where(keep)[0]; P = xyz[keep]
    return [(idx[i], idx[j]) for i in range(len(P)) for j in range(i + 1, len(P))
            if np.linalg.norm(P[i] - P[j]) < 1.95]

def draw(ax, crystal, docked, el, title):
    el = [str(e).strip().upper() for e in el]
    e1 = [e[:1] for e in el]; keep = [e != 'H' for e in e1]
    for a, b in bonds(crystal, e1): ax.plot(*zip(crystal[a], crystal[b]), color='#9ec3e6', lw=2, ls=(0, (4, 2)))
    for a, b in bonds(docked, e1):  ax.plot(*zip(docked[a], docked[b]), color='#666', lw=3)
    C, Dk = crystal[keep], docked[keep]; ek = [e for e, k in zip(el, keep) if k]
    ax.scatter(*C.T, c='#9ec3e6', s=45, alpha=.6, label='crystal')
    ax.scatter(*Dk.T, c=[EL.get(e, '#888') for e in ek], s=130, edgecolor='k', lw=.5, label='CIM pose')
    P = np.vstack([C, Dk]); c0 = P.mean(0); rad = np.abs(P - c0).max() + 1
    ax.set_xlim(c0[0]-rad, c0[0]+rad); ax.set_ylim(c0[1]-rad, c0[1]+rad); ax.set_zlim(c0[2]-rad, c0[2]+rad)
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.legend(fontsize=8, loc='upper left')

fig = plt.figure(figsize=(15, 5))
for i, p in enumerate(MOLS):
    d, r = BEST[p]
    ranked = backends._rank_unique(list(d["solutions"]), d["Q"].astype(float))
    lig = SimpleNamespace(coords=d["lig_coords"])
    poses, _ = _matches_to_poses(lig, d["variables"], d["box_coords"], ranked)
    best = np.array(poses)[int(r.argmin())]
    draw(fig.add_subplot(1, 3, i + 1, projection='3d'),
         d["lig_coords"], best, d["lig_elements"], f"{p}  —  {r.min():.2f} A  (paper {PAPER[p]})")
plt.tight_layout()
os.makedirs(os.path.join(ROOT, "assets"), exist_ok=True)
fig.savefig(os.path.join(ROOT, "assets", "docking_demo.png"), dpi=110, bbox_inches="tight")
plt.show()

/var/folders/0n/34k1_dz92mz91bw99965jzy40000gn/T/ipykernel_45944/725820225.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. A note on the classical solver

The same QUBO also runs on Kaiwu's CPU annealer,
`kw.classical.SimulatedAnnealingOptimizer` (wrapped as `backends.solve_sa`) — a
license-free, network-free sanity check. SA converges toward the QUBO's global
minimum; on these dense van-der-Waals matrices that minimum is a low-energy pose
*near* the crystal, while it is the CIM's high-variance sampling — pooled over runs —
that surfaces the lowest-RMSD (native) pose. That is exactly why the recipe pools
several CIM runs and reports the minimum-RMSD pose.

## References

- Wei *et al.*, "A versatile coherent Ising computing platform," *Light: Sci.
  Appl.* (2026) 15:74, DOI 10.1038/s41377-025-02178-1 — its molecular-docking
  application is reproduced here.
- Zha *et al.*, "Encoding Molecular Docking for Quantum Computers," *JCTC* (2024),
  DOI 10.1021/acs.jctc.3c00943 — the GPM/FAM QUBO encodings.

Kaiwu SDK © QBoson. AutoGrid, OpenBabel, Meeko under their own licenses.